<a href="https://colab.research.google.com/github/athitthiyan/Learning_Gen_AI/blob/main/Detection_Metrics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install bert-score transformers sentence-transformers scikit-learn torch openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.7 MB/s eta 0:00:00


In [2]:
pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 87.1 MB/s eta 0:00:00


In [3]:
import torch
from bert_score import score
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
import numpy as np

###########################################################
# SAMPLE DATA
###########################################################

question = "Who invented the telephone?"

generated_answer = """
Alexander Graham Bell invented the telephone in 1876.
He was awarded the first US patent for the device.
"""

reference_answer = """
Alexander Graham Bell is credited with inventing the telephone
and received the first US patent for it in 1876.
"""

documents = [
    "Alexander Graham Bell received the first US patent for the telephone in 1876.",
    "Bell is widely credited as the inventor of the telephone.",
    "The telephone revolutionized communication."
]

###########################################################
# 1. FACTSCORE (Retrieval-based factual precision)
###########################################################

embedder = SentenceTransformer('all-MiniLM-L6-v2')

def compute_factscore(answer, docs, threshold=0.55):
    """
    Simple FactScore approximation:
    - Split answer into sentences
    - Retrieve most similar document
    - Check semantic similarity
    """

    sentences = [s.strip() for s in answer.split(".") if s.strip()]

    doc_embeddings = embedder.encode(docs, convert_to_tensor=True)

    supported = 0

    for sent in sentences:
        sent_embedding = embedder.encode(sent, convert_to_tensor=True)

        similarities = util.cos_sim(sent_embedding, doc_embeddings)[0]
        best_score = similarities.max().item()

        print(f"\nSentence: {sent}")
        print(f"Best retrieval similarity: {best_score:.3f}")

        if best_score >= threshold:
            supported += 1

    factscore = supported / len(sentences)
    return factscore

factscore = compute_factscore(generated_answer, documents)

###########################################################
# 2. HHEM-like Hallucination Detection
###########################################################
# NOTE:
# Vectara's HHEM is proprietary.
# Here we approximate using NLI / contradiction detection.

classifier = pipeline(
    "text-classification",
    model="facebook/bart-large-mnli"
)

def detect_hallucination_nli(answer, docs):
    """
    Check whether answer contradicts evidence docs.
    """

    context = " ".join(docs)

    result = classifier(
        f"{context} </s></s> {answer}",
        candidate_labels=["supported", "hallucinated"]
    )

    return result

hhem_result = detect_hallucination_nli(generated_answer, documents)

###########################################################
# 3. BERTScore
###########################################################

P, R, F1 = score(
    [generated_answer],
    [reference_answer],
    lang="en",
    verbose=False
)

bertscore_f1 = F1.mean().item()

###########################################################
# 4. SelfCheckGPT
###########################################################

def selfcheck_gpt(generations):
    """
    Compute consistency between multiple generations.
    Higher similarity => lower hallucination probability
    """

    embeddings = embedder.encode(generations, convert_to_tensor=True)

    similarities = []

    for i in range(len(generations)):
        for j in range(i + 1, len(generations)):
            sim = util.cos_sim(
                embeddings[i],
                embeddings[j]
            ).item()

            similarities.append(sim)

    return np.mean(similarities)

###########################################################
# Example alternate generations
###########################################################

multiple_generations = [
    "Alexander Graham Bell invented the telephone in 1876.",
    "The telephone was invented by Alexander Graham Bell.",
    "Bell received the first patent for the telephone."
]

selfcheck_score = selfcheck_gpt(multiple_generations)

###########################################################
# RESULTS
###########################################################

print("\n==============================")
print("HALLUCINATION DETECTION REPORT")
print("==============================")

print(f"\nFactScore: {factscore:.3f}")

print(f"\nBERTScore F1: {bertscore_f1:.3f}")

print(f"\nSelfCheckGPT Consistency: {selfcheck_score:.3f}")

print("\nHHEM-like Result:")
print(hhem_result)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



HALLUCINATION DETECTION REPORT

FactScore: 1.000

BERTScore F1: 0.953

SelfCheckGPT Consistency: 0.846

HHEM-like Result:
[{'label': 'entailment', 'score': 0.9852452874183655}]
